# Remote Cleanup 03: Manual `atexit` Check With Kernel Restart

Focused manual test for orderly kernel shutdown. The seed cell creates live resources and does not manually release them. After restart, the verification cell checks whether `atexit` cleanup made the old IDs fail.

In [ ]:
import gc
import json
import os
from pathlib import Path

import numpy as np

REMOTE_IP = os.environ.get("PYNQ_REMOTE_DEVICES", "192.168.2.197").split(",")[0].strip()
os.environ["PYNQ_REMOTE_DEVICES"] = REMOTE_IP
OVERLAY_PATH = "/workspace/phd/PYNQ.remote-dev/applications/PYNQ/tests/resizer.xsa"

import pynq

print("REMOTE_IP:", REMOTE_IP)
print("OVERLAY_PATH:", OVERLAY_PATH)

In [ ]:
# Atexit setup: create live resources through normal PYNQ probing, then restart the kernel.
# Do not manually call cleanup or release. The point is to see whether orderly kernel restart runs atexit cleanup.
if hasattr(pynq.Device, "_active_device"):
    delattr(pynq.Device, "_active_device")
if hasattr(pynq.Device, "_devices"):
    delattr(pynq.Device, "_devices")
atexit_device = [d for d in pynq.Device.devices if d.has_capability("REMOTE")][0]
atexit_overlay = pynq.Overlay(OVERLAY_PATH, device=atexit_device)
atexit_resizer = atexit_overlay.resize_accel_0
atexit_resizer.read(0)
atexit_mmio = atexit_resizer.mmio
atexit_buffer = atexit_device.allocate(shape=(16,), dtype=np.uint32, cacheable=1)
atexit_buffer[:] = np.arange(16, dtype=np.uint32)
atexit_buffer.flush()

atexit_gpio = None
atexit_gpio_id = None
atexit_gpio_path = None
base_path = pynq.GPIO.get_gpio_base_path(device=atexit_device)
npins = pynq.GPIO.get_gpio_npins(device=atexit_device)
if base_path and npins:
    atexit_gpio_pin = pynq.GPIO.get_gpio_pin(0, device=atexit_device)
    atexit_gpio_path = f"/sys/class/gpio/gpio{atexit_gpio_pin}"
    if not atexit_device.exists_file(atexit_gpio_path).exists:
        atexit_gpio = pynq.GPIO(atexit_gpio_pin, "in", device=atexit_device)
        atexit_gpio.read()
        atexit_gpio_id = atexit_gpio._gpio_id

STATE_FILE = Path("/tmp/pynq_remote_cleanup_atexit_state.json")
state = {
    "mmio_id": atexit_mmio._remote_map.mmio_id,
    "buffer_id": atexit_buffer.buffer_id,
    "gpio_id": atexit_gpio_id,
    "gpio_path": atexit_gpio_path,
}
STATE_FILE.write_text(json.dumps(state, indent=2))
print(state)
print("Restart the kernel now. Watch for the final Cleanup Request in the target logs.")

In [ ]:
# Atexit verification using normal PYNQ discovery.
# Run Setup first after restart, then this cell.
STATE_FILE = Path("/tmp/pynq_remote_cleanup_atexit_state.json")
state = json.loads(STATE_FILE.read_text())
print("old handles saved before restart:", state)
print("Check the target logs for a Cleanup Request when the previous kernel restarted.")

if hasattr(pynq.Device, "_active_device"):
    delattr(pynq.Device, "_active_device")
if hasattr(pynq.Device, "_devices"):
    delattr(pynq.Device, "_devices")
device = [d for d in pynq.Device.devices if d.has_capability("REMOTE")][0]
print("fresh auto-cleanup device:", device)
print("If atexit did not clean the old resources, this constructor cleanup should clean them now.")

overlay = pynq.Overlay(OVERLAY_PATH, device=device)
resizer = overlay.resize_accel_0
print("fresh register read:", resizer.read(0))